# RetailMind AI — Day 4: Feature Engineering & Ensembles

Engineer domain features, evaluate feature selection and ensembles, compare against Day 3, and save the selected pipeline.

In [192]:
import json
from pathlib import Path
import joblib
import numpy as np
import pandas as pd
from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.impute import SimpleImputer
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
from sklearn.model_selection import StratifiedKFold, cross_val_score, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from xgboost import XGBClassifier

RANDOM_STATE = 42
DATA_PATH = Path('../data/Telco-Customer-Churn.csv')
REPORT_DIR = Path('../data/reports')
MODEL_DIR = Path('../models')
REPORT_DIR.mkdir(parents=True, exist_ok=True)
MODEL_DIR.mkdir(parents=True, exist_ok=True)

# Load and clean the data.
df = pd.read_csv(DATA_PATH).drop_duplicates().copy()
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')
df = df.loc[df['Churn'].isin(['Yes', 'No'])].copy()
df['Churn'] = df['Churn'].map({'No': 0, 'Yes': 1}).astype(int)

# Domain feature engineering.
df['tenure_bucket'] = pd.cut(df['tenure'], bins=[-1, 12, 24, 48, 72], labels=['0-12', '13-24', '25-48', '49-72'])
df['charge_ratio'] = df['TotalCharges'] / (df['MonthlyCharges'] * df['tenure'])
df['charge_ratio'] = df['charge_ratio'].replace([np.inf, -np.inf], np.nan)
df['is_month_to_month'] = (df['Contract'] == 'Month-to-month').astype(int)
df['has_online_security'] = (df['OnlineSecurity'] == 'Yes').astype(int)

X = df.drop(columns=['customerID', 'Churn'])
y = df['Churn']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, stratify=y, random_state=RANDOM_STATE)
print('Train:', X_train.shape, 'Test:', X_test.shape, 'Churn rate:', round(y.mean(), 4))

numeric_features = ['SeniorCitizen', 'tenure', 'MonthlyCharges', 'TotalCharges', 'charge_ratio', 'is_month_to_month', 'has_online_security']
categorical_features = ['gender', 'Partner', 'Dependents', 'PhoneService', 'MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract', 'PaperlessBilling', 'PaymentMethod', 'tenure_bucket']

preprocessor = ColumnTransformer([
    ('numeric', Pipeline([('imputer', SimpleImputer(strategy='median')), ('scaler', StandardScaler())]), numeric_features),
    ('categorical', Pipeline([('imputer', SimpleImputer(strategy='most_frequent')), ('encoder', OneHotEncoder(handle_unknown='ignore'))]), categorical_features)
])

def make_pipeline(classifier, select_features):
    steps = [('preprocessor', clone(preprocessor))]
    if select_features:
        steps.append(('feature_selection', SelectKBest(score_func=f_classif, k=30)))
    steps.append(('model', classifier))
    return Pipeline(steps)

scale_pos_weight = (y_train == 0).sum() / (y_train == 1).sum()
candidates = {
    'Random Forest': make_pipeline(RandomForestClassifier(n_estimators=300, min_samples_leaf=3, class_weight='balanced', random_state=RANDOM_STATE, n_jobs=-1), False),
    'Random Forest + SelectKBest': make_pipeline(RandomForestClassifier(n_estimators=300, min_samples_leaf=3, class_weight='balanced', random_state=RANDOM_STATE, n_jobs=-1), True),
    'XGBoost + SelectKBest': make_pipeline(XGBClassifier(n_estimators=300, max_depth=3, learning_rate=0.05, subsample=0.8, colsample_bytree=0.8, scale_pos_weight=scale_pos_weight, eval_metric='logloss', random_state=RANDOM_STATE, n_jobs=-1), True)
}
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

results, fitted_models = [], {}
for name, pipeline in candidates.items():
    cv_recall = cross_val_score(pipeline, X_train, y_train, scoring='recall', cv=cv, n_jobs=1)
    pipeline.fit(X_train, y_train)
    prediction = pipeline.predict(X_test)
    probability = pipeline.predict_proba(X_test)[:, 1]
    fitted_models[name] = pipeline
    results.append({'Model': name, 'CV Recall Mean': cv_recall.mean(), 'CV Recall Std': cv_recall.std(), 'Test Recall': recall_score(y_test, prediction), 'Test Precision': precision_score(y_test, prediction, zero_division=0), 'Test F1': f1_score(y_test, prediction, zero_division=0), 'Test Accuracy': accuracy_score(y_test, prediction), 'Test ROC-AUC': roc_auc_score(y_test, probability)})

results_df = pd.DataFrame(results).sort_values('CV Recall Mean', ascending=False).reset_index(drop=True)
day3_path = REPORT_DIR / 'churn_v3_report.json'
if day3_path.exists():
    day3 = json.loads(day3_path.read_text(encoding='utf-8'))
    day3_row = pd.DataFrame([{'Model': 'Day 3 SVM (recorded)', 'CV Recall Mean': day3['best_cv_recall'], 'CV Recall Std': np.nan, 'Test Recall': day3['test_metrics']['recall'], 'Test Precision': day3['test_metrics']['precision'], 'Test F1': day3['test_metrics']['f1'], 'Test Accuracy': day3['test_metrics']['accuracy'], 'Test ROC-AUC': day3['test_metrics']['roc_auc']}])
    comparison_df = pd.concat([day3_row, results_df], ignore_index=True)
else:
    comparison_df = results_df.copy()

winner_name = results_df.iloc[0]['Model']
winner = fitted_models[winner_name]
model_path = MODEL_DIR / 'churn_pipeline_day4.joblib'
joblib.dump(winner, model_path)
comparison_path = REPORT_DIR / 'day4_model_comparison.csv'
comparison_df.to_csv(comparison_path, index=False)
report = {'project': 'RetailMind AI', 'increment': 'Day 4', 'selection_metric': '5-fold stratified CV recall', 'production_model': winner_name, 'saved_model': str(model_path), 'engineered_features': ['tenure_bucket', 'charge_ratio', 'is_month_to_month', 'has_online_security'], 'feature_selection': 'SelectKBest(f_classif, k=30)', 'results': results_df.round(6).to_dict(orient='records')}
report_path = REPORT_DIR / 'churn_day4_report.json'
report_path.write_text(json.dumps(report, indent=2), encoding='utf-8')
print('\nModel comparison:')
display(comparison_df.round(4))
print('\nSelected production model:', winner_name)
print('Saved pipeline:', model_path)
print('Saved report:', report_path)

Train: (5634, 23) Test: (1409, 23) Churn rate: 0.2654

Model comparison:


,Model,CV Recall Mean,CV Recall Std,Test Recall,Test Precision,Test F1,Test Accuracy,Test ROC-AUC
0,Day 3 SVM (recorded),0.8355,NaN,0.8209,0.4575,0.5876,0.6941,0.8093
1,XGBoost + SelectKBest,0.7940,0.0266,0.7995,0.5120,0.6242,0.7445,0.8405
2,Random Forest + SelectKBest,0.7492,0.0179,0.7380,0.5370,0.6216,0.7615,0.8329
3,Random Forest,0.7378,0.0258,0.7460,0.5417,0.6277,0.7651,0.8394



Selected production model: XGBoost + SelectKBest
Saved pipeline: ..\models\churn_pipeline_day4.joblib
Saved report: ..\data\reports\churn_day4_report.json
